This notebook contains a few selected problems from my Computational Number Theory Course

Worksheet 1

In [526]:
print(is_prime(5))

True


In [527]:
# The Sieve of Eratosthenes
def sieve(N):
    '''
    Returns a list of all prime numbers up to N.
    '''
    A,B = list(range(2,N+1)),list()
    for i in range(2,round(sqrt(N))+1):
        for n in A:             # Circling the prime number
            if n%i==0:
                A.remove(n)
                B.append(i)
                break
        for n in A:             # Removing all its multiples
            if n%i==0: A.remove(n)
    return(B+A)

print(sieve(1000))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311, 313, 317, 331, 337, 347, 349, 353, 359, 367, 373, 379, 383, 389, 397, 401, 409, 419, 421, 431, 433, 439, 443, 449, 457, 461, 463, 467, 479, 487, 491, 499, 503, 509, 521, 523, 541, 547, 557, 563, 569, 571, 577, 587, 593, 599, 601, 607, 613, 617, 619, 631, 641, 643, 647, 653, 659, 661, 673, 677, 683, 691, 701, 709, 719, 727, 733, 739, 743, 751, 757, 761, 769, 773, 787, 797, 809, 811, 821, 823, 827, 829, 839, 853, 857, 859, 863, 877, 881, 883, 887, 907, 911, 919, 929, 937, 941, 947, 953, 967, 971, 977, 983, 991, 997]


Worksheet 2

In [528]:
# Basic Euclidean algorithm for integers.
def EA(a,b):
    r = [a,b]
    while(r[-1]!=0):
        r.append(r[-2]%r[-1])
    return(r[-2])

a,b = 100,8
print(EA(a,b))

4


In [529]:
# The Extended Euclidean algorithm for integers.
def EEA(a,b):
    r = [a,b]
    s = [1,0]
    t = [0,1]
    while(r[-1]!=0):
        s.append(s[-2]-(r[-2]//r[-1])*s[-1])
        t.append(t[-2]-(r[-2]//r[-1])*t[-1])
        r.append(r[-2]%r[-1])
    return(r[-2],s[-2],t[-2])

a,b = 100,8
print(EEA(a,b))
d,s,t = EEA(a,b)
print(a*s+b*t==d)   # Checking correctness

# alternatively
xgcd(a,b)

(4, 1, -12)
True


(4, 1, -12)

In [530]:
# Euclidean algorithm for polynomials over F_p
F = GF(5)
R.<x> = PolynomialRing(F)
A = x^5 + 2*x^4 + 4*x + 1
B = 4*x + 1

def EA_p(A,B):
    R = [A,B]
    while(R[-1].degree()!=0):
        R.append(R[-2].quo_rem(R[-1])[1])
        # R.append(divmod(R[-2],R[-1])[1]) #alternatively
    return R


print(EA_p(A,B))
print(A.gcd(B))

[x^5 + 2*x^4 + 4*x + 1, 4*x + 1, 3]
1


Worksheet 3

In [531]:
a = 6
n = 31
print(EEA(a,n)[1]%n)    # modular inverse

# alternatively
xgcd(a,n)[1]%n

26


26

In [532]:
%%timeit
for i in range(1000):
    a = 6
    n = 31
    EEA(a,n)[1]%n    # modular inverse

4.1 ms ± 361 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [533]:
def modular_linear_equation(a,b,n):
    """returns solution to ax=bmodn"""
    return((xgcd(a,n)[1]*b)%n)
modular_linear_equation(4,6,31)

17

Worksheet 4

In [534]:
# Matrix multiplication using the Chinese Remainder Theorem
def modular_matrix_product(A, B):
    if A.ncols() != B.nrows():
        raise ValueError('Incompatible dimensions for matrix multiplication')

    max_entry = max(abs(x) for x in A.list() + B.list())
    d = A.ncols()
    bound = d * max_entry * max_entry

    modulus = 1
    primes = []
    p = 2
    while modulus <= 2*bound:
        primes.append(p)
        modulus *= p
        p = next_prime(p)

    C_mods = []
    for p in primes:
        Zp = Integers(p)
        A_mod = A.change_ring(Zp)
        B_mod = B.change_ring(Zp)
        C_mods.append(A_mod * B_mod)

    C = matrix(ZZ, A.nrows(), B.ncols())
    for i in range(A.nrows()):
        for j in range(B.ncols()):
            residues = [int(C_mods[k][i,j]) for k in range(len(primes))]
            x = crt(residues, primes)
            if x > modulus//2 : x -= modulus
            C[i,j] = x

    return C

A = matrix(ZZ,[[1, 4],[5, 2]])
B = matrix(ZZ,[[1, 0],[0, 1]])
C = modular_matrix_product(A, B)
print('Reconstructed product:')
print(C)
print('Direct product:')
print(A * B)

Reconstructed product:
[1 4]
[5 2]
Direct product:
[1 4]
[5 2]


In [535]:
def bruteforce_twosquares(p):
    '''Checks if a prime number can be expressed as a sum of two squares and returns the pair if it exists'''
    if is_prime(p):
        for i in range(p//2):
            if(frac(sqrt(i+1)) + frac(sqrt(p-i-1))==0):return True,i+1,p-i-1
    return False
bruteforce_twosquares(17)

(True, 1, 16)

In [536]:
# Fermat’s two square algorithm using effective Thue’s lemma.
import random

def EEA(a,b):
    r = [a,b]
    s = [1,0]
    t = [0,1]
    while(r[-1]!=0):
        s.append(s[-2]-(r[-2]//r[-1])*s[-1])
        t.append(t[-2]-(r[-2]//r[-1])*t[-1])
        r.append(r[-2]%r[-1])
    return(r,t)

def twosquares(p):
    if(p%4!=1 or p.is_prime(p)!=True):return(False)
    e = (p-1)/4
    U = list(range(1,p))
    flag = False
    while(flag == False):
        k = random.choice(U)
        b = (k^e)%p 
        if((b*b)%p == p-1):flag = True
    l =EEA(p,b)
    for i in range(len(l[0])):
        if(l[0][i] < sqrt(p)):return(abs(l[0][i])%p,abs(l[1][i])%p)
            
print(twosquares(17))

(4, 1)


Worksheet 5

In [537]:
def modexp(x,y,n):
    '''returns x^y mod n'''
    if y==0: return 1
    z = modexp(x,y//2,n)
    if y%2==0: return (z*z)%n
    else: return (x*z*z)%n

In [538]:
# RSA Key Generation
import random
p=random_prime(10^6,lbound=10^5)
q=random_prime(10^6,lbound=10^5)
print(p,q)
n = p*q
phi = (p-1)*(q-1)
e = random.randint(2,phi-1)
while(gcd(e,phi)>1):
    e = random.randint(2,phi-1)
d = (xgcd(e,phi)[1])%phi
print((e*d)%phi)

def encrypt(message,n,e):
    return modexp(message,e,n)

def decrypt(message,n,d):
    return modexp(message,d,n)

message = 4
encrypted_message = encrypt(message,n,e)
decrypted_message = decrypt(encrypted_message,n,d)
print(decrypted_message)

411679 263083
1
4


In [539]:
def jacobi(n,k):
    if (k<=0 or k%2==0):return False
    n = n%k
    t=1
    while(n!=0):
        while(n%2==0):
            n=n/2
            r=k%8
            if r==3 or r==5: t =-t
        n,k = k,n
        if (n%4 == 3 and k%4 == 3): t = -t
        n = n%k
    if k==1: return t
    else: return 0

print(jacobi(5,11))

1


Worksheet 6

In [540]:
def twopow(p):
    '''returns h and q such that p = 2^h * q where q is odd'''
    h=0
    while(p%2==0):
        h+=1
        p/=2
    return(h,p)

print(twopow(3137-1))

(6, 49)


In [541]:
def powsearch(h, start, stop, n):
    low, high = start, stop
    while low <= high:
        mid = (low + high) // 2
        if mid^h <= n: low = mid + 1
        else: high = mid - 1
    return high

def isperfectpower(n):
    '''returns k,h such that n = k^h if n is a perfect power, else returns (n,1) and does it in O(log(n)^2) time'''
    l = len(bin(n))- 2
    for h in range(2,l):
        k = powsearch(h,2,n,n)
        # k = floor(n^(1/h)) #alternatively
        if k^h == n: return k,h
    return (n,1)

isperfectpower(36)


(6, 2)

Worksheet 7

In [542]:
def SimpleBabyGiantStep(p,a):
    R = IntegerModRing(p)
    g = R.multiplicative_generator()
    m = floor(sqrt(p))
    table = {(g**i)%p : i for i in range(0,m)}
    gamma = g**m.inverse_mod(p)
    b,j,i = a,0,-1
    if b in table.keys(): i = table[b]
    while i==-1:
        b = b*gamma
        j = j+1
        if b in table.keys(): i = table[b]
    return (j*m + i)

SimpleBabyGiantStep(3137,45)


1391

Worksheet 8

In [543]:
import random
p = 17
C = [i for i in range(1,p)]
a = random.choice(C)
while(modexp(a,(p-1)//2,p)==1):
    a = random.choice(C)
print(a,modexp(a,(p-1)//2,p))

7 16


In [544]:
import random
def factorize_pq(n):
    '''Las Vegas algorithm to factor n=pq'''
    C = [i+1 for i in range(n-1)]
    a = random.choice(C)
    d = gcd(a,n)
    if d != 1: return(d)
    return(False)    
print(factorize_pq(15))

False


In [545]:
# Converts the previous code block into a Monte Carlo algorithm
n = 15
flag = factorize_pq(n)
while(not flag):
    flag = factorize_pq(n)
print(flag)

3


In [546]:
def random_sqroot(a,p):
    '''Monte Carlo algorithm to find a square root of a mod p'''
    if(power_mod(a,(p-1)//2,p)!=1):return False
    C = [i for i in range(1,p)]
    x = random.choice(C)
    while(0 != (power_mod(x,2,p)-a)%p):
        x = random.choice(C)
    return(x)
print(random_sqroot(4,17))    

2


Worksheet 9

In [547]:
def fermat_test(n,a):return not bool(power_mod(a,n-1,n)-1)
# n = 2
# while(not (fermat_test(n,2) or is_prime(n))):n = n+1
# print(n)

In [548]:
import random

def gcd_prim(n,k):
    '''
    GCD primality test. Returns True if n is probably prime, False if n is composite.
    '''
    L = [i for i in range(1,n)]
    for i in range(k):
        a = random.choice(L)
        if gcd(a,n)>1: return False
    return True

def solovay_strassen(n,k):
    '''
    Solvay-Strassen primality test. Returns True if n is probably prime, False if n is composite.
    '''
    if n % 2 == 0: return n == 2
    L = [i for i in range(1,n)]
    for i in range(k):
        a = random.choice(L)
        if gcd(a,n)>1: return False
        p = (n-1)/2
        d = int(power_mod(a,p,n) - kronecker(a,n))
        if d%n!=0: return False
    return True

n,g,s = 10000,0,0
for i in range(2,n):
    if gcd_prim(i,1)==is_prime(i):g = g+1
    if solovay_strassen(i,1) == is_prime(i):s = s+1
print(g/(n-1),s/(n-1))

5092/9999 9976/9999


Worksheet 10

In [549]:
p = 5
F = GF(p)
a = F(2)
b = F(1)
E = EllipticCurve(F,[a,b])
print(E)

Elliptic Curve defined by y^2 = x^3 + 2*x + 1 over Finite Field of size 5


In [550]:
E.points()

[(0 : 1 : 0),
 (0 : 1 : 1),
 (0 : 4 : 1),
 (1 : 2 : 1),
 (1 : 3 : 1),
 (3 : 2 : 1),
 (3 : 3 : 1)]

In [551]:
O = E(0)
P = E(1,2)
Q = E(3,3)
R = 4*P+2*Q
print(R)
P.order()

(1 : 2 : 1)


7

In [552]:
E.abelian_group()

Additive abelian group isomorphic to Z/7 embedded in Abelian group of points on Elliptic Curve defined by y^2 = x^3 + 2*x + 1 over Finite Field of size 5

In [553]:
E = EllipticCurve(GF(41),[2,5])
print(E.abelian_group())
print(E.abelian_group().invariants())

Additive abelian group isomorphic to Z/22 + Z/2 embedded in Abelian group of points on Elliptic Curve defined by y^2 = x^3 + 2*x + 5 over Finite Field of size 41
(2, 22)


In [554]:
P = E(1,7)
T = E(10,0)
from sage.groups.additive_abelian.additive_abelian_wrapper import AdditiveAbelianGroupWrapper
H = AdditiveAbelianGroupWrapper.from_generators([P])
print(T in H)

False


In [555]:
p = 1097393683
print(is_prime(p))
E = EllipticCurve(GF(p), [1,0])
print(E.cardinality())
print(E.abelian_group().invariants())

True
1097393684
(1097393684,)
